# Walk-forward: OLS, XGBoost and hybrid

One run directory per configuration (training window x feature set), all model legs inside it
(`partial/<leg>/<symbol>.parquet`, merged into `daily_diagnostics.parquet` with a `model_leg` column).
Per configuration: random search on the tuning block for the 5 tuning stocks -> pooled winner per
horizon frozen into the manifest (D026) -> walk-forward for every leg and stock on the days after the
block (D027). The tuning block is the same for every configuration, so all windows and legs are
evaluated on identical test days; training windows may reach back into the block.

Checkpoints are one file per (stock, target): `trials/<sym>/<target>.parquet` and
`partial/<leg>/<sym>/<target>.parquet`. Re-running with more stocks or horizons computes only the
missing units; the manifest is rewritten with the current lists (frozen `params` are kept and extended).

`OFFSET` moves the tuning block later in the sample (D018 stability check); the default 0 is the design.

In [ ]:
import os, sys, json, time
from datetime import datetime, timezone

import numpy as np
import pandas as pd

sys.path.append(os.path.dirname(os.getcwd()))
import importlib
import utils.data_processing as du
import utils.execution as execution
import utils.pipeline as pipeline
import utils.workers as workers

for m in (du, pipeline, execution, workers):
    importlib.reload(m)

In [ ]:
# Config
LOCAL = True   # local subset: 2 stocks, ~20 days

RUN_SYMBOLS = du.HEADLINE_SYMBOLS   # walk-forward stocks (20-stock stratified sample; du.SYMBOLS = all 42)
TUNE_SYMBOLS = du.TUNE_SYMBOLS      # tuning stocks
HORIZONS = ["100ms", "1s", "2s", "5s", "15s", "30s", "1m", "2.5m", "5m"]

# one run per (training window, feature set)
CONFIGS = [
    dict(train_days=1, features=du.STANDARD_FEATURES),
    dict(train_days=5, features=du.STANDARD_FEATURES),
    dict(train_days=20, features=du.STANDARD_FEATURES),
    dict(train_days=10, features=du.STANDARD_FEATURES),
]
LEGS = ["ols", "xgb", "hybrid"]   # hybrid = OLS + booster on the OLS train residuals

TUNE_BLOCK_LEN = 25   # >= max train_days + N_PAIRS; identical for all configs -> identical test days
OFFSET = 0.0          # tuning-block start as a share of the sample (0 = design; 0.50 = stability check)

N_TRIALS = 40
N_PAIRS = 5
SEED = 0

N_PROC = 5            # stocks in parallel
N_JOBS = 2            # CPU threads per process

# search space: shallow trees, low learning rates
SEARCH_SPACE = {
    "max_depth": ("int", 1, 4),
    "learning_rate": ("log", 0.01, 0.1),
    "min_child_weight": ("logint", 5, 100),
    "subsample": ("uniform", 0.5, 1.0),
    "colsample_bytree": ("uniform", 0.5, 1.0),
    "reg_lambda": ("log", 0.01, 10.0),
}
EARLY_STOPPING = dict(n_estimators=2000, early_stopping_rounds=50, eval_metric="rmse")
XGB_PARAMS = dict(tree_method="hist", max_bin=128, n_jobs=N_JOBS, random_state=0)

if LOCAL:
    RUN_SYMBOLS = TUNE_SYMBOLS = ["Adidas", "Qiagen"]
    CONFIGS = [dict(train_days=1, features=du.STANDARD_FEATURES)]
    HORIZONS = ["100ms", "2s", "5m"]
    TUNE_BLOCK_LEN, N_TRIALS, N_PAIRS, N_PROC = 6, 4, 2, 2

PARENT = os.path.dirname(os.getcwd())
OUTPUT_ROOT = f"{PARENT}/model_outputs"
DEVICE = execution.select_device()

ALL_DATES = list(du.SAMPLE_DATES)
start = round(OFFSET * len(ALL_DATES))
TUNE_DATES = ALL_DATES[start:start + TUNE_BLOCK_LEN]
EVAL_DATES = ALL_DATES[start + TUNE_BLOCK_LEN:]
assert TUNE_BLOCK_LEN >= max(c["train_days"] for c in CONFIGS) + N_PAIRS, "tuning block too short for the longest window"
assert len(EVAL_DATES) >= 2, "no evaluation days after the tuning block"

print(f"device: {DEVICE}\ntuning: {TUNE_DATES[0]} .. {TUNE_DATES[-1]} ({len(TUNE_DATES)} days)"
      f"\nevaluation: {EVAL_DATES[0]} .. {EVAL_DATES[-1]} ({len(EVAL_DATES)} test days for every config)")

In [ ]:
# tune -> freeze -> walk-forward, one run dir per config; every step skips finished stocks
def update_manifest(run_dir, **fields):
    with open(f"{run_dir}/manifest.json") as f:
        manifest = json.load(f)
    manifest.update(fields)
    with open(f"{run_dir}/manifest.json", "w") as f:
        json.dump(manifest, f, indent=2, default=str)
    return manifest


GLOBAL_START = time.perf_counter()
for cfg in CONFIGS:
    name = pipeline.run_name(cfg["train_days"], cfg["features"])
    if OFFSET > 0:
        name += f"_off{int(OFFSET * 100)}"
    run_dir = f"{OUTPUT_ROOT}/runs/{name}"
    feature_cols, target_cols = workers.feature_target_cols(RUN_SYMBOLS[0], HORIZONS, cfg["features"])

    pipeline.start_run(OUTPUT_ROOT, name, {
        "status": "tuning",
        "purpose": "",
        "symbols": RUN_SYMBOLS,
        "tune_symbols": TUNE_SYMBOLS,
        "features": cfg["features"],
        "horizons": HORIZONS,
        "feature_cols": feature_cols,
        "target_cols": target_cols,
        "train_days": cfg["train_days"],
        "dates": ALL_DATES,
        "tune_dates": TUNE_DATES,
        "offset": OFFSET,
        "legs": LEGS,
        "xgb_params": XGB_PARAMS,
        "target_scale": pipeline.TARGET_SCALE,
        "tuning": {"seed": SEED, "n_trials": N_TRIALS, "n_pairs": N_PAIRS,
                   "search_space": SEARCH_SPACE, "early_stopping": EARLY_STOPPING},
        "params": None,
    })

    # tuning on the tuning stocks (workers skip finished targets); frozen params of earlier horizons are kept
    execution.run_parallel(workers.tune_xgb, {s: (run_dir, s, DEVICE) for s in TUNE_SYMBOLS}, n_proc=N_PROC)
    all_trials = workers.load_units(run_dir, "trials", TUNE_SYMBOLS, target_cols)
    with open(f"{run_dir}/manifest.json") as f:
        stored = json.load(f)["params"] or {}
    update_manifest(run_dir, params={**workers.freeze_winners(all_trials, SEARCH_SPACE), **stored}, status="running")

    # walk-forward, leg by leg (workers skip finished (stock, target) units)
    for leg in LEGS:
        codes = execution.run_parallel(workers.walk_forward, {s: (run_dir, s, leg, DEVICE) for s in RUN_SYMBOLS}, n_proc=N_PROC)
        print(f"{name} {leg}: {time.perf_counter()-GLOBAL_START:.0f}s, failed: {[s for s, c in codes.items() if c]}")

    daily = pd.concat([workers.load_units(run_dir, f"partial/{leg}", RUN_SYMBOLS, target_cols) for leg in LEGS], ignore_index=True)
    daily.to_parquet(f"{run_dir}/daily_diagnostics.parquet", index=False)
    manifest = update_manifest(run_dir, status="complete")
    update_manifest(run_dir, runtime_seconds=round((datetime.now(timezone.utc) - datetime.fromisoformat(manifest["created_at"])).total_seconds(), 1))
    print(f"{name}: done, {time.perf_counter()-GLOBAL_START:.0f}s elapsed")

In [ ]:
# Quick look: mean MSE ratio per leg x horizon for each config
for cfg in CONFIGS:
    name = pipeline.run_name(cfg["train_days"], cfg["features"]) + (f"_off{int(OFFSET * 100)}" if OFFSET > 0 else "")
    daily = pd.read_parquet(f"{OUTPUT_ROOT}/runs/{name}/daily_diagnostics.parquet")
    daily["horizon"] = pd.Categorical(daily["target"].map(workers.horizon_of), categories=HORIZONS, ordered=True)
    print(name)
    display(daily.pivot_table(index="model_leg", columns="horizon", values="mse_ratio", aggfunc="mean", observed=True).round(4))